# Bayesian Optimization on the best tested model — NASA Battery B0005

The ablation runs told us which configuration performs best. This notebook takes that winner and
tunes it with Bayesian search, then places the result in the benchmark table against the
published works.

**Step 1** reads `ablation_battery_b0005_results_v3.json` (or the v2 file if v3 has not been run)
and selects the best-scoring configuration across every block. Its architecture — whether the
LSTM is bidirectional, whether attention pooling is used, whether the CNN is multi-scale, which
feature set, which target and which split — is **fixed**, and only the hyperparameters are
searched.

**Step 2** runs 30 Bayesian trials over that architecture's filters, kernel sizes, LSTM units,
dense layers, dropout rates, optimizer and learning rate.

**Step 3** builds the benchmark table.

Set `FORCE_CONFIG` in the second cell if you would rather nominate the architecture yourself
instead of taking whichever the ablation picked.

Run from the folder containing `B0005.mat`. Roughly 20-30 minutes for one architecture.

In [ ]:
import os, json, gc, time, glob
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, Softmax, Multiply, Lambda,
                                     GlobalMaxPooling1D, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('op determinism unavailable:', e)
print('TF', tf.__version__, '| keras-tuner', kt.__version__,
      '| GPU', tf.config.list_physical_devices('GPU'))

## Step 1 — which model won the ablation

`A0`-`A5` are the one-step ladder, `B0`-`B3` the recursive one. The mapping below turns an
ablation row id into the architecture flags that produced it.

In [ ]:
MAT_PATH = 'B0005.mat'
WINDOW   = 10
MAX_TRIALS, SEEDS_TRIAL, SEEDS_FINAL = 30, 3, 10
SEARCH_EPOCHS, SEARCH_PATIENCE = 150, 20
FINAL_EPOCHS,  FINAL_PATIENCE  = 300, 30
BATCH, SEED = 8, 42

# set this to override the automatic pick, e.g.
# FORCE_CONFIG = dict(bidirectional=True, attention=True, multiscale=True,
#                     variant='base', target='delta', split=80, label='manual')
FORCE_CONFIG = None

FLAGS = {   # ablation id -> architecture switches
    'A0': dict(bidirectional=False, attention=False, multiscale=False),
    'A1': dict(bidirectional=False, attention=False, multiscale=False),
    'A2': dict(bidirectional=True,  attention=False, multiscale=False),
    'A3': dict(bidirectional=True,  attention=True,  multiscale=False),
    'A4': dict(bidirectional=True,  attention=True,  multiscale=True),
    'A5': dict(bidirectional=True,  attention=True,  multiscale=True),
    'B0': dict(bidirectional=False, attention=False, multiscale=False),
    'B1': dict(bidirectional=True,  attention=False, multiscale=False),
    'B2': dict(bidirectional=True,  attention=True,  multiscale=False),
    'B3': dict(bidirectional=True,  attention=True,  multiscale=True),
}
BLOCKS = {  # json key -> (target, split, protocol)
    'ablation_level':    ('level', 50, 'onestep'),
    'ablation_delta':    ('delta', 50, 'onestep'),
    'ablation_split80':  ('level', 80, 'onestep'),
    'block1_one_step':   ('delta', 50, 'onestep'),
    'block2_recursive':  ('delta', 80, 'recursive'),
}

def pick_best():
    if FORCE_CONFIG: 
        print('using FORCE_CONFIG'); return FORCE_CONFIG
    src = None
    for f in ('ablation_battery_b0005_results_v3.json', 'ablation_battery_b0005_results_v2.json'):
        if os.path.exists(f): src = f; break
    if src is None:
        raise FileNotFoundError('no ablation results file found — run the ablation notebook '
                                'first, or set FORCE_CONFIG')
    d = json.load(open(src)); print('reading', src)
    cands = []
    for bk, (tgt, sp, proto) in BLOCKS.items():
        for r in d.get(bk, []):
            v = r.get('mean', r.get('rmse'))
            if v is None: continue
            cands.append(dict(block=bk, id=r['id'], config=r['config'],
                              variant=r.get('variant', 'base'), rmse=float(v),
                              target=tgt, split=sp, protocol=proto))
    if not cands: raise ValueError('no scored rows in %s' % src)
    tb = pd.DataFrame(cands).sort_values('rmse').reset_index(drop=True)
    print('\nall tested configurations, best first:\n')
    print(tb.head(12).to_string(index=False))
    b = tb.iloc[0]
    cfg = dict(FLAGS[b['id']]); cfg.update(variant=b['variant'], target=b['target'],
                                           split=int(b['split']), protocol=b['protocol'],
                                           label='%s / %s' % (b['block'], b['id']),
                                           ablation_rmse=float(b['rmse']))
    return cfg

BEST = pick_best()
print('\n' + '='*72)
print('OPTIMIZING: %s  (ablation RMSE %.5f)' % (BEST['label'], BEST.get('ablation_rmse', float('nan'))))
print('  bidirectional %s | attention %s | multi-scale %s' %
      (BEST['bidirectional'], BEST['attention'], BEST['multiscale']))
print('  features %s | target %s | split %d | protocol %s' %
      (BEST['variant'], BEST['target'], BEST['split'], BEST['protocol']))
print('='*72)

In [ ]:
from scipy.io import loadmat
mat = loadmat(MAT_PATH); cycles = mat['B0005'][0, 0]['cycle'][0]
rows, n = [], 0
for i in range(len(cycles)):
    r = cycles[i]
    if r['type'][0] != 'discharge': continue
    n += 1; d = r['data'][0][0]
    V = d['Voltage_measured'][0].astype(float); T = d['Temperature_measured'][0].astype(float)
    t = d['Time'][0].astype(float)
    tt = lambda vth: (float(t[np.where(V <= vth)[0][0]]) if len(np.where(V <= vth)[0]) else float(t[-1]))
    rows.append(dict(cycle=n, capacity=float(d['Capacity'][0][0]),
                     discharge_time=float(t[-1]), t_to_3p8V=tt(3.8), t_to_3p5V=tt(3.5),
                     mean_voltage=float(V.mean()), min_voltage=float(V.min()),
                     mean_temp=float(T.mean()), max_temp=float(T.max()),
                     temp_rise=float(T.max()-T[0])))
HI = pd.DataFrame(rows); C0 = HI['capacity'].iloc[0]
HI['SOH'] = (HI['capacity']/C0).astype(np.float32)
S_ALL, CYC = HI['SOH'].values.astype(np.float32), HI['cycle'].values
BASE_COLS = ['capacity']; FULL_COLS = [c for c in HI.columns if c not in ('cycle','SOH')]
print('cycles %d | initial capacity %.4f Ah' % (len(HI), C0))

cols = BASE_COLS if BEST['variant'] == 'base' else FULL_COLS
ref  = HI[cols].iloc[0].replace(0, 1.0)
FEAT = (HI[cols]/ref).values.astype(np.float32)
NF   = FEAT.shape[1]

Xtr, ytr, ptr, Xte, yte, pte = [], [], [], [], [], []
for e in range(WINDOW, len(HI)):
    seq = FEAT[e-WINDOW:e]
    yv  = S_ALL[e]-S_ALL[e-1] if BEST['target'] == 'delta' else S_ALL[e]
    if CYC[e] < BEST['split']: Xtr.append(seq); ytr.append(yv); ptr.append(S_ALL[e-1])
    else:                      Xte.append(seq); yte.append(yv); pte.append(S_ALL[e-1])
f = lambda a: np.asarray(a, np.float32); Xtr, ytr, ptr = f(Xtr), f(ytr), f(ptr)
cut = int(0.75*len(Xtr))
D = dict(Xf=Xtr[:cut], yf=ytr[:cut], Xv=Xtr[cut:], yv=ytr[cut:], pv=ptr[cut:],
         Xt=f(Xte), yt=f(yte), pt=f(pte),
         val_start=WINDOW+cut, test_start=int(np.where(CYC >= BEST['split'])[0][0]))
print('features %d | fit %s val %s test %s' % (NF, D['Xf'].shape, D['Xv'].shape, D['Xt'].shape))

## Step 2 — Bayesian search

The architecture switches are fixed at the ablation winner. Only the sizes, dropouts, optimizer
and learning rate are searched — the same tunables as the N-CMAPSS Bayesian notebook, with the
lower bounds on filters and units extended from 32 down to 8 because B0005 supplies only a few
dozen training windows.

Each trial is scored as the mean of 3 seeds: one architecture re-run with a different seed
varies by several fold on this dataset, so a single-seed objective would let the search settle on
a lucky run.

In [ ]:
GI = lambda s: tf.keras.initializers.GlorotUniform(seed=s)

def attention_pool(seq, s):
    w = Softmax(axis=1)(Dense(1, activation='tanh', kernel_initializer=GI(s))(seq))
    return Lambda(lambda x: K.sum(x, axis=1))(Multiply()([seq, w]))

def build_model(hp, seed=SEED):
    B = BEST
    inp = Input(shape=(WINDOW, NF)); x = inp
    # ---- CNN branch (multi-scale if the winning configuration used it) ----
    nc = hp.Int('num_cnn_layers', 1, 3)
    c = x
    for i in range(nc):
        fl = hp.Int('cnn_%d_filters' % i, 8, 128, step=8)
        if B['multiscale']:
            c = Concatenate()([Conv1D(fl, k, padding='same', activation='relu',
                                      kernel_initializer=GI(seed))(c) for k in (3, 5, 7)])
        else:
            c = Conv1D(fl, hp.Int('cnn_%d_kernel' % i, 3, 7, step=2), padding='same',
                       activation='relu', kernel_initializer=GI(seed))(c)
        c = Dropout(hp.Float('cnn_%d_dropout' % i, 0.0, 0.3, step=0.1), seed=seed)(c)
        if i < 2 and int(c.shape[1]) > 2: c = MaxPooling1D(2, padding='same')(c)
    c = (GlobalMaxPooling1D() if hp.Choice('cnn_pooling', ['max','avg']) == 'max'
         else GlobalAveragePooling1D())(c)
    # ---- LSTM branch ----
    nl = hp.Int('num_lstm_layers', 1, 2)
    r = x
    for i in range(nl):
        last = (i == nl-1)
        cell = LSTM(hp.Int('lstm_%d_units' % i, 8, 128, step=8),
                    return_sequences=(not last) or B['attention'],
                    dropout=hp.Float('lstm_%d_dropout' % i, 0.0, 0.3, step=0.1),
                    recurrent_dropout=hp.Float('lstm_%d_rdropout' % i, 0.0, 0.3, step=0.1),
                    kernel_initializer=GI(seed))
        r = Bidirectional(cell)(r) if B['bidirectional'] else cell(r)
    if B['attention']: r = attention_pool(r, seed)
    # ---- head ----
    h = Concatenate()([c, r])
    for i in range(hp.Int('num_dense_layers', 1, 3)):
        h = Dense(hp.Int('dense_%d_units' % i, 8, 256, step=8), activation='relu',
                  kernel_initializer=GI(seed))(h)
        h = Dropout(hp.Float('dense_%d_dropout' % i, 0.0, 0.5, step=0.1), seed=seed)(h)
    out = Dense(1, activation='linear', kernel_initializer=GI(seed))(h)
    lr  = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4, 1e-5])
    opt = {'adam': tf.keras.optimizers.Adam, 'rmsprop': tf.keras.optimizers.RMSprop,
           'sgd': tf.keras.optimizers.SGD}[hp.Choice('optimizer', ['adam','rmsprop','sgd'])](lr)
    m = Model(inp, out); m.compile(optimizer=opt, loss='mse', metrics=['mae']); return m

In [ ]:
def rmse_onestep(m, part):
    p = m.predict(D['X'+part], batch_size=512, verbose=0).ravel()
    prev, y = D['p'+part], D['y'+part]
    lvl  = prev + p if BEST['target'] == 'delta' else p
    true = prev + y if BEST['target'] == 'delta' else y
    return float(np.sqrt(mean_squared_error(true, lvl)))

def rollout(m, start, n):
    win, out = list(FEAT[start-WINDOW:start]), []
    lvl_prev = S_ALL[start-1]
    for _ in range(n):
        x = np.asarray(win[-WINDOW:], np.float32).reshape(1, WINDOW, NF)
        p = float(m(x, training=False).numpy().ravel()[0])
        lvl = float(np.clip(lvl_prev + p if BEST['target'] == 'delta' else p, 0.0, 1.2))
        out.append(lvl)
        step = win[-1].copy(); step[0] = lvl      # feed the predicted capacity back
        win.append(step); lvl_prev = lvl
    return np.asarray(out, np.float32)

def rmse_recursive(m, part):
    st, n = ((D['val_start'], len(D['Xv'])) if part == 'v'
             else (D['test_start'], len(HI)-D['test_start']))
    return float(np.sqrt(mean_squared_error(S_ALL[st:st+n], rollout(m, st, n))))

def score(m, part):
    return (rmse_onestep if BEST['protocol'] == 'onestep' else rmse_recursive)(m, part)

def persistence(part='t'):
    if BEST['protocol'] == 'onestep':
        prev, y = D['p'+part], D['y'+part]
        true = prev + y if BEST['target'] == 'delta' else y
        return float(np.sqrt(mean_squared_error(true, prev)))
    st = D['test_start']; n = len(HI)-st
    return float(np.sqrt(mean_squared_error(S_ALL[st:], np.full(n, S_ALL[st-1]))))

def train_one(m, ep, pat):
    m.fit(D['Xf'], D['yf'], validation_data=(D['Xv'], D['yv']), epochs=ep, batch_size=BATCH,
          verbose=0, shuffle=True,
          callbacks=[EarlyStopping('val_loss', patience=pat, restore_best_weights=True),
                     ReduceLROnPlateau('val_loss', factor=0.5, patience=8, min_lr=1e-7)])
    return m

OBJ = 'val_rmse_mean'
class MultiSeedBO(kt.BayesianOptimization):
    def run_trial(self, trial, *a, **kw):
        hp, sc = trial.hyperparameters, []
        for s in range(SEEDS_TRIAL):
            K.clear_session(); gc.collect()
            np.random.seed(SEED+s); tf.random.set_seed(SEED+s)
            try:
                v = score(train_one(build_model(hp, SEED+s), SEARCH_EPOCHS, SEARCH_PATIENCE), 'v')
                if not np.isfinite(v): v = 1e3
            except Exception as e:
                print('   seed %d failed (%s)' % (s, type(e).__name__)); v = 1e3
            sc.append(v)
        print('   trial %s: %.5f (std %.5f)' % (trial.trial_id, np.mean(sc), np.std(sc, ddof=1)))
        return {OBJ: float(np.mean(sc))}

In [ ]:
t0 = time.time()
tuner = MultiSeedBO(hypermodel=build_model,
                    objective=kt.Objective(OBJ, direction='min'),
                    max_trials=MAX_TRIALS, seed=SEED,
                    directory='bo_best_battery', project_name='best_model', overwrite=False)
tuner.search()
best_hp = tuner.get_best_hyperparameters(1)[0]
print('\nsearch finished in %.0f s' % (time.time()-t0))

scores, params = [], None
for s in range(SEEDS_FINAL):
    K.clear_session(); gc.collect()
    sd = SEED + 1000 + s; np.random.seed(sd); tf.random.set_seed(sd)
    m = build_model(best_hp, sd); params = m.count_params()
    train_one(m, FINAL_EPOCHS, FINAL_PATIENCE)
    scores.append(score(m, 't'))

RES = dict(config=BEST, params=int(params), best_hp=dict(best_hp.values),
           per_seed=[round(v, 5) for v in scores], mean=float(np.mean(scores)),
           std=float(np.std(scores, ddof=1)), best=float(np.min(scores)),
           persistence=persistence(), seconds=round(time.time()-t0))
print('\n' + '='*72)
print('RESULT  %.5f +/- %.5f SOH   (%.4f Ah)' % (RES['mean'], RES['std'], RES['mean']*C0))
print('best seed %.5f | persistence %.5f | params %s'
      % (RES['best'], RES['persistence'], format(params, ',')))
print('ablation score before tuning: %.5f  ->  after tuning: %.5f'
      % (BEST.get('ablation_rmse', float('nan')), RES['mean']))
print('='*72)
print('\nselected hyperparameters')
for k, v in sorted(RES['best_hp'].items()): print('   %-22s %s' % (k, v))

## Step 3 — benchmark table

Published values are on the normalised SOH scale, so ours are converted from Ah by dividing by
the 1.8565 Ah initial capacity. Tian and Gao forecast recursively from a start point, so those
rows only compare like-for-like with a recursive result — the `protocol` column makes clear which
of ours is which.

In [ ]:
PUBLISHED = [
    ('CNN-BiLSTM-AM',   'Tian et al., Batteries 2022 [53]', 0.00481, 'recursive'),
    ('CNN-BiLSTM',      'Tian et al. [53]',                 0.00595, 'recursive'),
    ('BiLSTM',          'Tian et al. [53]',                 0.00620, 'recursive'),
    ('1D CNN-BiLSTM',   'Gao et al., Meas. & Control 2023', 0.00772, 'recursive'),
    ('CNN',             'Tian et al. [53]',                 0.00863, 'recursive'),
    ('1D CNN',          'Gao et al.',                       0.00866, 'recursive'),
    ('1D CNN-LSTM',     'Gao et al.',                       0.00932, 'recursive'),
    ('1D CNN-RNN',      'Gao et al.',                       0.01661, 'recursive'),
    ('RNN',             'Gao et al.',                       0.01776, 'recursive'),
    ('BiLSTM',          'Gao et al.',                       0.01986, 'recursive'),
    ('LSTM',            'Gao et al.',                       0.02302, 'recursive'),
    ('CL-Net',          'Khan et al., Mathematics 2021',    0.205,   'not stated'),
    ('Encoder-decoder', 'Khan et al.',                      0.221,   'not stated'),
    ('CNN-LSTM',        'Khan et al.',                      0.247,   'not stated'),
    ('LSTM',            'Khan et al.',                      0.259,   'not stated'),
]
rows  = [dict(method=m, source=s, rmse_SOH=round(v, 5), protocol=p) for m, s, v, p in PUBLISHED]
rows += [dict(method='Ours: Parallel CNN-LSTM (as published)', source='this work',
              rmse_SOH=round(0.047/1.8565, 5), protocol='one-step'),
         dict(method='Ours: Sequential + Hyperband (as published)', source='this work',
              rmse_SOH=round(0.094/1.8565, 5), protocol='one-step'),
         dict(method='Ours: Sequential (as published)', source='this work',
              rmse_SOH=round(0.350/1.8565, 5), protocol='one-step')]
rows += [dict(method='Ours: best tested model + Bayesian (%s)' % BEST['label'],
              source='this work (new)', rmse_SOH=round(RES['mean'], 5),
              protocol=BEST['protocol']),
         dict(method='   persistence floor', source='reference',
              rmse_SOH=round(RES['persistence'], 5), protocol=BEST['protocol'])]

bench = pd.DataFrame(rows)
pd.set_option('display.width', 200)
for pr in ('recursive', 'one-step', 'not stated'):
    sub = bench[bench.protocol == pr].sort_values('rmse_SOH').reset_index(drop=True)
    if len(sub):
        print('\n' + '='*88); print('PROTOCOL: %s' % pr.upper()); print('='*88)
        print(sub.to_string(index=False))
bench.sort_values(['protocol','rmse_SOH']).to_csv('benchmark_battery_best_model_bayesian.csv',
                                                  index=False)
json.dump(RES, open('bayesian_best_model_battery.json', 'w'), indent=2, default=str)
print('\nsaved benchmark_battery_best_model_bayesian.csv and bayesian_best_model_battery.json')